# Chapter 12 - Case Outstanding Development Technique

> "The case outstanding development technique is used to estimate future development on known claims (i.e., case outstanding development). The method can also be used to estimate total unpaid claims, including both development on known claims and pure IBNR... In this chapter, we refer to this technique as Approach #1. We also present an example for a self-insurer of the standard development technique applied to case outstanding in which the case outstanding development factors to ultimate are derived from industry-based benchmark reported and paid claim development factors; we refer to this example as Approach #2."
> -- Friedland, Chapter 12

This notebook reconciles the exhibits from Friedland Chapter 12:
- **Exhibit I**: U.S. Industry Auto (Approach #1) - Sheets 1 through 5
- **Exhibit II**: XYZ Insurer - Auto BI (Approach #1) - Sheets 1 through 7
- **Exhibit III**: Self-Insurer Case Outstanding Only - General Liability (Approach #2)


In [1]:
import numpy as np
import pandas as pd
import chainladder as cl
import warnings
from IPython.display import display as nb_display

# Suppress minor warnings for clean presentation
warnings.filterwarnings("ignore")


def format_exh(
    exh_df: pd.DataFrame,
    value_cols: list[str] = [],
    factor_cols: list[str] = [],
    other_formats: dict = {},
    origin_format: str = "{:%Y}",
    origin_name: str = "Accident Year",
    total_cols: list[str] = [],
):
    """
    Actuarial exhibit styler matching Chapter 11.
    Adds column numbering (1), (2), ..., currency/factor formatting, and summary totals.
    """

    def _format(
        df: pd.DataFrame,
        val_c: list[str] = [],
        fac_c: list[str] = [],
        oth_f: dict = {},
        orig_f: str = None,
        orig_n: str = origin_name,
    ):
        idx_name = df.index.name if df.index.name else "index"
        format_dict = {}
        if orig_f:
            format_dict[orig_n] = orig_f
        format_dict.update({x: "{:,.0f}" for x in val_c})
        format_dict.update({x: "{:,.3f}" for x in fac_c})
        format_dict.update(oth_f)

        return (
            df
            .reset_index()
            .rename(columns={idx_name: orig_n})
            .style.hide(axis="index")
            .set_properties(**{"text-align": "right"})
            .format(format_dict)
        )

    col_idx = pd.DataFrame(
        [[f"({i + 2})" for i in range(len(exh_df.columns))]],
        columns=exh_df.columns,
        index=["(1)"],
    )
    table_styler = _format(col_idx, orig_f=None, orig_n=origin_name).concat(
        _format(
            exh_df,
            val_c=value_cols,
            fac_c=factor_cols,
            oth_f=other_formats,
            orig_f=origin_format,
            orig_n=origin_name,
        )
    )

    if len(total_cols) > 0:
        table_styler = table_styler.concat(
            _format(
                exh_df.agg(["sum"]).rename(index={"sum": "Total"}),
                val_c=value_cols,
                fac_c=factor_cols,
                oth_f={x: "" for x in exh_df.columns if x not in total_cols},
                orig_f=None,
                orig_n=origin_name,
            )
        )
    return table_styler

## Exhibit I Analysis

In Exhibit I, Friedland demonstrates **Approach #1** using the U.S. Industry Auto dataset (`usauto`). This technique estimates unpaid claims by analyzing the historical relationships of incremental paid claims and case reserve runoff to prior carried case reserves.

Following the design pattern established in **Chapter 11**, we fit our dedicated `cl.CaseOutstanding` models upfront to power the subsequent exhibits.


## Exhibit I Sheet 1

### Case Outstanding and Incremental Paid Claims ($000)

- **Case Outstanding**: Calculated dynamically as $\text{Case Outstanding} = \text{Reported Claims} - \text{Paid Claims}$.
- **Incremental Paid Claims**: Derived dynamically from Cumulative Paid Claims via `.cum_to_incr()`.


In [ ]:
usauto = cl.load_sample("usauto")
paid_cumulative = usauto["paid"]
incurred = usauto["incurred"]

# 1. Triangle of Case Outstanding ($000)
case_outstanding = incurred - paid_cumulative

# 2. Triangle of Incremental Paid Claims ($000)
incremental_paid = paid_cumulative.cum_to_incr()

# - m_all: fits with all historical origin periods for review
# - m_5: fits 5-year averages
# - m_3: fits 3-year averages (used for selected ratios)
# - m_medial: fits 5-year medial (5x1) averages dropping highest and lowest ratios
m_all = cl.CaseOutstanding(
    paid_to_incurred=("paid", "incurred"), paid_n_periods=-1, case_n_periods=-1
).fit(usauto)
m_5 = cl.CaseOutstanding(
    paid_to_incurred=("paid", "incurred"), paid_n_periods=5, case_n_periods=5
).fit(usauto)
m_3 = cl.CaseOutstanding(
    paid_to_incurred=("paid", "incurred"), paid_n_periods=3, case_n_periods=3
).fit(usauto)
m_medial = cl.CaseOutstanding(
    paid_to_incurred=("paid", "incurred"),
    paid_n_periods=5,
    paid_drop_high=1,
    paid_drop_low=1,
    case_n_periods=5,
    case_drop_high=1,
    case_drop_low=1,
).fit(usauto)

print("PART 1 - Case Outstanding Triangle as of (months) ($000):")
nb_display(case_outstanding.style.format("{:,.0f}", na_rep="").highlight_diagonal())

print("\nPART 2 - Incremental Paid Claims Triangle as of (months) ($000):")
nb_display(incremental_paid.style.format("{:,.0f}", na_rep="").highlight_diagonal())

## Exhibit I Sheet 2

### Ratio of Incremental Paid Claims to Previous Case Outstanding

This sheet presents:
- **PART 1 - Ratio Triangle**: Evaluated natively via `cl.CaseOutstanding.paid_to_prior_case_`, calculating the proportion of previous case reserves paid out during each subsequent development lag:
  $$\text{Ratio}_{d} = \frac{\text{Incremental Paid}_{d}}{\text{Case Outstanding}_{d-12}}$$
- **PART 2 - Historical Averages**: Summary table combining the Latest 5 Years (`m_5.paid_ldf_`), Latest 3 Years (`m_3.paid_ldf_`), and 5-Year Medial (5x1).
- **PART 3 - Selected Ratios**: Actuarial selections based on the Latest 3 Years average with **To Ult = 1.100**.


In [ ]:
print(
    "PART 1 - Ratio of Incremental Paid Claims to Previous Case Outstanding (cl.Triangle):"
)
tri_paid = m_all.paid_to_prior_case_
nb_display(tri_paid.round(3))

print("\nPART 2 - Historical Averages of the Ratio:")
averages_paid_df = pd.DataFrame({
    "Latest 5 Years": m_5.paid_ldf_.to_frame().iloc[0],
    "Latest 3 Years": m_3.paid_ldf_.to_frame().iloc[0],
    "5-Year Medial (5x1)": m_medial.paid_ldf_.to_frame().iloc[0],
}).T
averages_paid_df["To Ult"] = np.nan
nb_display(averages_paid_df.style.format("{:.3f}", na_rep=""))

print(
    "\nPART 3 - Selected Ratio of Incremental Paid Claims to Previous Case Outstanding:"
)
dev_cols_e1 = [24, 36, 48, 60, 72, 84, 96, 108, 120]
selected_paid_ratios = pd.Series(m_3.paid_ldf_.values[0, 0, 0, :], index=dev_cols_e1)
selected_paid_ratios["To Ult"] = 1.100
selected_paid_df = pd.DataFrame([selected_paid_ratios], index=["Selected"])
nb_display(selected_paid_df.style.format("{:.3f}", na_rep=""))

## Exhibit I Sheet 3

### Ratio of Case Outstanding to Previous Case Outstanding

This sheet presents:
- **PART 1 - Ratio Triangle**: Evaluated natively via `cl.CaseOutstanding.case_to_prior_case_`, calculating the proportion of case reserves remaining after each subsequent development lag:
  $$\text{Ratio}_{d} = \frac{\text{Case Outstanding}_{d}}{\text{Case Outstanding}_{d-12}}$$
- **PART 2 - Historical Averages**: Summary table combining the Latest 5 Years (`m_5.case_ldf_`), Latest 3 Years (`m_3.case_ldf_`), and 5-Year Medial (5x1).
- **PART 3 - Selected Ratios**: Actuarial selections based on the Latest 3 Years average with **To Ult = 0.000**.


In [ ]:
print("PART 1 - Ratio of Case Outstanding to Previous Case Outstanding (cl.Triangle):")
tri_case = m_all.case_to_prior_case_
nb_display(tri_case.round(3))

print("\nPART 2 - Historical Averages of the Ratio:")
averages_case_df = pd.DataFrame({
    "Latest 5 Years": m_5.case_ldf_.to_frame().iloc[0],
    "Latest 3 Years": m_3.case_ldf_.to_frame().iloc[0],
    "5-Year Medial (5x1)": m_medial.case_ldf_.to_frame().iloc[0],
}).T
averages_case_df["To Ult"] = np.nan
nb_display(averages_case_df.style.format("{:.3f}", na_rep=""))

print("\nPART 3 - Selected Ratio of Case Outstanding to Previous Case Outstanding:")
selected_case_ratios = pd.Series(m_3.case_ldf_.values[0, 0, 0, :], index=dev_cols_e1)
selected_case_ratios["To Ult"] = 0.000
selected_case_df = pd.DataFrame([selected_case_ratios], index=["Selected"])
nb_display(selected_case_df.style.format("{:.3f}", na_rep=""))

## Exhibit I Sheet 4

### Projection of Paid Claims ($000)

Using the selected ratios:
1. **Complete the Square of Case Outstanding**:
   $$\text{Case}_{d} = \text{Case}_{d-12} \times \text{Selected Case Ratio}_{d}$$
2. **Complete the Square of Incremental Paid Claims**:
   $$\text{Incr Paid}_{d} = \text{Case}_{d-12} \times \text{Selected Paid Ratio}_{d}$$
   $$\text{Incr Paid}_{\text{To Ult}} = \text{Case}_{120} \times \text{Selected Paid Ratio}_{\text{To Ult}}$$
3. **Cumulative Paid Claims**: Cumulative sum across development lags through `To Ult`.

`chainladder` natively completes both the case outstanding runoff and cumulative paid projections via `cl.Chainladder().fit(model.fit_transform(triangle))`. Projected cells in the lower triangle are highlighted natively using `highlight_lower_triangle()`.

In [ ]:
# Native CaseOutstanding projection with Chainladder
cl_fit_e1 = cl.Chainladder().fit(m_3.fit_transform(usauto))
full_paid_e1 = cl_fit_e1.full_triangle_["paid"]
full_incurred_e1 = cl_fit_e1.full_triangle_["incurred"]
full_case_e1 = full_incurred_e1 - full_paid_e1
full_incr_paid_e1 = full_paid_e1.cum_to_incr()

print("1. Case Outstanding as of (months) ($000) [Projected cells highlighted]:")
nb_display(
    full_case_e1
    .iloc[..., :10]
    .style.format("{:,.0f}", na_rep="")
    .highlight_lower_triangle()
)

print(
    "\n2. Incremental Paid Claims as of (months) ($000) [Projected cells highlighted]:"
)
nb_display(
    full_incr_paid_e1
    .iloc[..., :10]
    .style.format("{:,.0f}", na_rep="")
    .highlight_lower_triangle()
)

print("\n3. Cumulative Paid Claims as of (months) ($000):")
nb_display(
    full_paid_e1
    .iloc[..., :10]
    .style.format("{:,.0f}", na_rep="")
    .highlight_lower_triangle()
)

## Exhibit I Sheet 5

### Development of Unpaid Claim Estimate ($000)

Reconciles the unpaid claim estimate across all accident years using the actuarial exhibit styler (`format_exh`) from **Chapter 11**:
- **(2) Reported Claims**: Latest diagonal of reported claims.
- **(3) Paid Claims**: Latest diagonal of cumulative paid claims.
- **(4) Projected Ultimate**: Cumulative paid claims at `To Ult`.
- **(5) Case Outstanding**: $(2) - (3)$.
- **(6) Estimated IBNR**: $(4) - (2)$.
- **(7) Total Unpaid**: $(4) - (3)$.


In [ ]:
case_120_e1 = full_case_e1.iloc[..., 9]
ult_paid_e1 = (
    (full_paid_e1.iloc[..., 9] + case_120_e1 * 1.100)
    .to_frame(origin_as_datetime=False)
    .iloc[:, 0]
)

sheet5_df = pd.DataFrame(
    index=case_outstanding.to_frame(origin_as_datetime=False).index
)
sheet5_df["Reported Claims"] = (
    incurred.latest_diagonal.to_frame(origin_as_datetime=False).iloc[:, 0].values
)
sheet5_df["Paid Claims"] = (
    paid_cumulative.latest_diagonal.to_frame(origin_as_datetime=False).iloc[:, 0].values
)
sheet5_df["Projected Ultimate"] = ult_paid_e1.values
sheet5_df["Case Outstanding"] = sheet5_df["Reported Claims"] - sheet5_df["Paid Claims"]
sheet5_df["Estimated IBNR"] = (
    sheet5_df["Projected Ultimate"] - sheet5_df["Reported Claims"]
)
sheet5_df["Total Unpaid"] = sheet5_df["Projected Ultimate"] - sheet5_df["Paid Claims"]
sheet5_df.index = pd.to_datetime(sheet5_df.index.astype(str) + "-12-31")

print("Development of Unpaid Claim Estimate ($000):")
nb_display(
    format_exh(
        sheet5_df,
        value_cols=list(sheet5_df.columns),
        total_cols=list(sheet5_df.columns),
        origin_name="Accident Year",
        origin_format="{:%Y}",
    )
)

---

## Exhibit II Analysis

Exhibit II applies the Case Outstanding Development Technique (Approach #1) to the **XYZ Insurer - Auto BI** data.
Because of changing claims operations, settlement rates, and case reserve adequacy noted in previous chapters, the actuary selects the **Latest 2** year averages for development factors, reflecting the most recent operating environment.


## Exhibit II Sheet 1

### Case Outstanding and Incremental Paid Claims ($000)

Source data is loaded from `cl.load_sample("friedland_xyz_auto_bi")`.
- **Case Outstanding**: $\text{Reported Claims} - \text{Paid Claims}$.
- **Incremental Paid Claims**: Computed via `.cum_to_incr()`.


In [ ]:
xyz = cl.load_sample("friedland_xyz_auto_bi")
xyz_rep = xyz["Reported Claims"]
xyz_paid = xyz["Paid Claims"]
xyz_case = xyz_rep - xyz_paid
xyz_incr_paid = xyz_paid.cum_to_incr()

# Fit CaseOutstanding models upfront
xyz_m_all = cl.CaseOutstanding(
    paid_to_incurred=("Paid Claims", "Reported Claims"),
    paid_n_periods=-1,
    case_n_periods=-1,
).fit(xyz)
xyz_m_5 = cl.CaseOutstanding(
    paid_to_incurred=("Paid Claims", "Reported Claims"),
    paid_n_periods=5,
    case_n_periods=5,
).fit(xyz)
xyz_m_2 = cl.CaseOutstanding(
    paid_to_incurred=("Paid Claims", "Reported Claims"),
    paid_n_periods=2,
    case_n_periods=2,
).fit(xyz)
xyz_m_medial = cl.CaseOutstanding(
    paid_to_incurred=("Paid Claims", "Reported Claims"),
    paid_n_periods=5,
    paid_drop_high=1,
    paid_drop_low=1,
    case_n_periods=5,
    case_drop_high=1,
    case_drop_low=1,
).fit(xyz)

print("Case Outstanding as of (months) ($000):")
nb_display(xyz_case.style.format("{:,.0f}", na_rep="").highlight_diagonal())

print("\nIncremental Paid Claims as of (months) ($000):")
nb_display(xyz_incr_paid.style.format("{:,.0f}", na_rep="").highlight_diagonal())

## Exhibit II Sheet 2

### Ratio of Incremental Paid Claims to Previous Case Outstanding

Calculates the ratio of incremental payments at age $d$ to case outstanding at age $d-12$:
$$\text{Ratio}_{d} = \frac{\text{Incremental Paid}_{d}}{\text{Case Outstanding}_{d-12}}$$

- **Latest 2 Average**: Reflects recent settlement velocity changes.
- **Selected**: Matches the **Latest 2** average across all maturities, with **To Ult = 1.100**.


In [ ]:
dev_cols_e2 = [24, 36, 48, 60, 72, 84, 96, 108, 120, 132]

tri_paid_xyz = xyz_m_all.paid_to_prior_case_

xyz_averages_paid_df = pd.DataFrame({
    "Latest 5": xyz_m_5.paid_ldf_.to_frame().iloc[0],
    "Latest 2": xyz_m_2.paid_ldf_.to_frame().iloc[0],
    "Latest 5x1": xyz_m_medial.paid_ldf_.to_frame().iloc[0],
}).T
xyz_averages_paid_df["To Ult"] = np.nan

xyz_selected_paid = pd.Series(xyz_m_2.paid_ldf_.values[0, 0, 0, :], index=dev_cols_e2)
xyz_selected_paid["To Ult"] = 1.100
xyz_selected_paid_df = pd.DataFrame([xyz_selected_paid], index=["Selected"])

print("Ratio of Incremental Paid Claims to Previous Case Outstanding:")
nb_display(tri_paid_xyz.round(3))

print(
    "\nAverages of the Ratio of Incremental Paid Claims to Previous Case Outstanding:"
)
nb_display(xyz_averages_paid_df.style.format("{:.3f}", na_rep=""))

print("\nSelected Ratio of Incremental Paid Claims to Previous Case Outstanding:")
nb_display(xyz_selected_paid_df.style.format("{:.3f}", na_rep=""))

## Exhibit II Sheet 3

### Ratio of Case Outstanding to Previous Case Outstanding

Calculates the runoff of case reserves:
$$\text{Ratio}_{d} = \frac{\text{Case Outstanding}_{d}}{\text{Case Outstanding}_{d-12}}$$

- **Latest 2 Average**: Captures recent case reserve adequacy strengthening.
- **Selected**: Matches the **Latest 2** average, with maturity 132 and **To Ult = 0.000**.


In [ ]:
tri_case_xyz = xyz_m_all.case_to_prior_case_

xyz_averages_case_df = pd.DataFrame({
    "Latest 5": xyz_m_5.case_ldf_.to_frame().iloc[0],
    "Latest 2": xyz_m_2.case_ldf_.to_frame().iloc[0],
    "Latest 5x1": xyz_m_medial.case_ldf_.to_frame().iloc[0],
}).T
xyz_averages_case_df["To Ult"] = np.nan

xyz_selected_case = pd.Series(xyz_m_2.case_ldf_.values[0, 0, 0, :], index=dev_cols_e2)
xyz_selected_case[132] = 0.000
xyz_selected_case["To Ult"] = 0.000
xyz_selected_case_df = pd.DataFrame([xyz_selected_case], index=["Selected"])

print("Ratio of Case Outstanding to Previous Case Outstanding:")
nb_display(tri_case_xyz.round(3))

print("\nAverages of the Ratio of Case Outstanding to Previous Case Outstanding:")
nb_display(xyz_averages_case_df.style.format("{:.3f}", na_rep=""))

print("\nSelected Ratio of Case Outstanding to Previous Case Outstanding:")
nb_display(xyz_selected_case_df.style.format("{:.3f}", na_rep=""))

## Exhibit II Sheet 4

### Projection of Paid Claims ($000)

Using the selected ratios for XYZ Insurer:
1. Complete square of Case Outstanding through age 132 and `To Ult` (0.0).
2. Complete square of Incremental Paid Claims through age 132 and `To Ult` ($1.100 \times \text{Case}_{132}$).
3. Cumulative Paid Claims through `To Ult`.

`chainladder` natively completes the squares and projects ultimate claims via `cl.Chainladder().fit(xyz_m_2.fit_transform(xyz))`.

In [ ]:
# Native CaseOutstanding projection with Chainladder
cl_fit_xyz = cl.Chainladder().fit(xyz_m_2.fit_transform(xyz))
xyz_full_paid = cl_fit_xyz.full_triangle_["Paid Claims"]
xyz_full_incurred = cl_fit_xyz.full_triangle_["Reported Claims"]
xyz_full_case = xyz_full_incurred - xyz_full_paid
xyz_full_incr_paid = xyz_full_paid.cum_to_incr()

print("1. Case Outstanding as of (months) ($000) [Projected cells highlighted]:")
nb_display(
    xyz_full_case
    .iloc[..., :11]
    .style.format("{:,.0f}", na_rep="")
    .highlight_lower_triangle()
)

print(
    "\n2. Incremental Paid Claims as of (months) ($000) [Projected cells highlighted]:"
)
nb_display(
    xyz_full_incr_paid
    .iloc[..., :11]
    .style.format("{:,.0f}", na_rep="")
    .highlight_lower_triangle()
)

print("\n3. Cumulative Paid Claims as of (months) ($000):")
nb_display(
    xyz_full_paid
    .iloc[..., :11]
    .style.format("{:,.0f}", na_rep="")
    .highlight_lower_triangle()
)

## Exhibit II Sheet 5

### Development of Unpaid Claim Estimate ($000)

Reconciles the unpaid claim estimate for XYZ Insurer using the `format_exh` styler:
- **(2) Reported Claims**: Latest diagonal of reported claims.
- **(3) Paid Claims**: Latest diagonal of cumulative paid claims.
- **(4) Projected Ultimate**: Cumulative paid claims through `To Ult`.
- **(5) Case Outstanding**: $(2) - (3)$.
- **(6) Estimated IBNR**: $(4) - (2)$.
- **(7) Total Unpaid**: $(4) - (3)$.

In [ ]:
xyz_sheet5_df = pd.DataFrame(index=xyz_case.to_frame(origin_as_datetime=False).index)
xyz_sheet5_df["Reported Claims"] = (
    xyz_rep.latest_diagonal.to_frame(origin_as_datetime=False).iloc[:, 0].values
)
xyz_sheet5_df["Paid Claims"] = (
    xyz_paid.latest_diagonal.to_frame(origin_as_datetime=False).iloc[:, 0].values
)
xyz_sheet5_df["Projected Ultimate"] = (
    cl_fit_xyz
    .ultimate_["Paid Claims"]
    .to_frame(origin_as_datetime=False)
    .iloc[:, 0]
    .values
)
xyz_sheet5_df["Case Outstanding"] = (
    xyz_sheet5_df["Reported Claims"] - xyz_sheet5_df["Paid Claims"]
)
xyz_sheet5_df["Estimated IBNR"] = (
    xyz_sheet5_df["Projected Ultimate"] - xyz_sheet5_df["Reported Claims"]
)
xyz_sheet5_df["Total Unpaid"] = (
    xyz_sheet5_df["Projected Ultimate"] - xyz_sheet5_df["Paid Claims"]
)
xyz_sheet5_df.index = pd.to_datetime(xyz_sheet5_df.index.astype(str) + "-12-31")

print("Development of Unpaid Claim Estimate ($000):")
nb_display(
    format_exh(
        xyz_sheet5_df,
        value_cols=list(xyz_sheet5_df.columns),
        total_cols=list(xyz_sheet5_df.columns),
        origin_name="Accident Year",
        origin_format="{:%Y}",
    )
)

## Exhibit II Sheet 6

### Summary of Ultimate Claims ($000)

Comparison of Projected Ultimate Claims across methods:
- **(4) & (5) Dev Method (Reported / Paid)**: Developed in Chapter 7, Exhibit II, Sheet 3.
- **(6) Expected Claims**: Developed in Chapter 8, Exhibit III, Sheet 1.
- **(7) Bornhuetter-Ferguson (Reported)**: Developed in Chapter 9, Exhibit II, Sheet 2.
- **(8) Bornhuetter-Ferguson (Paid)**: Developed in Chapter 9, Exhibit II, Sheet 1.
- **(9) Cape Cod (Reported)**: Developed in Chapter 10, Exhibit II, Sheet 2.
- **(10) Cape Cod (Paid)**: Developed in Chapter 10, Exhibit II, Sheet 1.
- **(11) & (12) Frequency-Severity (Approaches 1 & 2)**: Developed in Chapter 11, Exhibits II & IV.
- **(13) Case Outstanding Dev (Approach 1)**: Developed in Sheet 5 above.


In [ ]:
# Benchmark Ultimates from prior chapters
prior_ultimates = {
    "Dev Method (Reported) (4)": [
        16824,
        21876,
        26978,
        30725,
        36247,
        43764,
        48312,
        45155,
        43906,
        42964,
        41249,
    ],
    "Dev Method (Paid) (5)": [
        16824,
        21876,
        26978,
        30725,
        36247,
        42522,
        46114,
        43444,
        42125,
        41042,
        39686,
    ],
    "Expected Claims (6)": [
        16824,
        21876,
        26978,
        30725,
        36247,
        43534,
        47713,
        44026,
        46765,
        46682,
        47708,
    ],
    "BF (Reported) (7)": [
        16824,
        21876,
        26978,
        30725,
        36247,
        43764,
        48259,
        45057,
        44299,
        44018,
        44320,
    ],
    "BF (Paid) (8)": [
        16824,
        21876,
        26978,
        30725,
        36247,
        43105,
        47040,
        43813,
        44211,
        43632,
        44081,
    ],
    "Cape Cod (Reported) (9)": [
        16824,
        21876,
        26978,
        30725,
        36247,
        43764,
        48261,
        45063,
        44327,
        44093,
        44520,
    ],
    "Cape Cod (Paid) (10)": [
        16824,
        21876,
        26978,
        30725,
        36247,
        43075,
        46995,
        43743,
        44078,
        43376,
        43575,
    ],
    "Freq-Sev Approach 1 (11)": [
        16824,
        21876,
        26978,
        30725,
        36247,
        43764,
        47672,
        44589,
        43905,
        44435,
        44062,
    ],
    "Freq-Sev Approach 2 (12)": [
        16824,
        21876,
        26978,
        30725,
        36247,
        43764,
        47447,
        45391,
        46144,
        46274,
        45781,
    ],
}

summary_ult_df = pd.DataFrame(prior_ultimates, index=xyz_sheet5_df.index)
summary_ult_df["Case O/S Dev (13)"] = xyz_sheet5_df["Projected Ultimate"].values

print("Summary of Ultimate Claims ($000):")
nb_display(
    format_exh(
        summary_ult_df,
        value_cols=list(summary_ult_df.columns),
        total_cols=list(summary_ult_df.columns),
        origin_name="Accident Year",
        origin_format="{:%Y}",
    )
)

## Exhibit II Sheet 7

### Summary of Estimated IBNR ($000)

Estimated IBNR across methods: $\text{IBNR} = \text{Projected Ultimate} - \text{Reported Claims}$.


In [ ]:
summary_ibnr_df = pd.DataFrame(index=xyz_sheet5_df.index)
rep_diag = xyz_rep.latest_diagonal.to_frame(origin_as_datetime=False).iloc[:, 0].values

for col in summary_ult_df.columns:
    summary_ibnr_df[col] = summary_ult_df[col] - rep_diag

print("Summary of Estimated IBNR ($000):")
nb_display(
    format_exh(
        summary_ibnr_df,
        value_cols=list(summary_ibnr_df.columns),
        total_cols=list(summary_ibnr_df.columns),
        origin_name="Accident Year",
        origin_format="{:%Y}",
    )
)

---

## Exhibit III Analysis

In Exhibit III, Friedland demonstrates **Approach #2 (Implied Case Development Factors)** for self-insurers or lines where historical case runoff triangles are unavailable, using benchmark reported and paid claim development patterns.


## Exhibit III Sheet 1

### Self-Insurer Case Outstanding Only ($)

Approach #2 derives case outstanding development factors to ultimate from industry benchmark reported and paid claim development patterns.

#### Actuarial Formulation (Friedland p. 235)
Let $f_{\text{rep}}(d)$ and $f_{\text{paid}}(d)$ be the cumulative development factors to ultimate from maturity $d$.
The percentage of ultimate claims reported and paid at maturity $d$ are:
$$\%\text{Reported}(d) = \frac{1}{f_{\text{rep}}(d)}, \quad \%\text{Paid}(d) = \frac{1}{f_{\text{paid}}(d)}$$

The proportion of ultimate claims remaining in case outstanding is:
$$\%\text{Case}(d) = \%\text{Reported}(d) - \%\text{Paid}(d) = \frac{1}{f_{\text{rep}}(d)} - \frac{1}{f_{\text{paid}}(d)}$$

Since unpaid claims equal $1 - \%\text{Paid}(d)$, the ratio of unpaid claims to case outstanding (the **Case Outstanding CDF**) is:
$$\text{Case CDF}(d) = \frac{1 - \%\text{Paid}(d)}{\%\text{Case}(d)} = \frac{1 - \frac{1}{f_{\text{paid}}(d)}}{\frac{1}{f_{\text{rep}}(d)} - \frac{1}{f_{\text{paid}}(d)}} = \frac{(f_{\text{rep}}(d) - 1) \times f_{\text{paid}}(d)}{f_{\text{paid}}(d) - f_{\text{rep}}(d)} + 1.0$$

The estimated unpaid claims are then directly computed as:
$$\text{Estimated Unpaid Claims} = \text{Case Outstanding} \times \text{Case CDF}$$

`chainladder` natively supports this technique via the `cl.ImpliedCaseDevelopment` estimator, which derives the implied case development patterns and attaches them to the case triangle for downstream projection with `cl.Chainladder()`.

In [14]:
# Self-Insurer Case Outstanding Only - General Liability (Exhibit III)
exh3_years = [1998, 1999, 2000, 2001, 2002, 2003]
exh3_case = [500000, 650000, 800000, 850000, 975000, 1000000]

# Construct case outstanding triangle for self-insurer at Dec 31, 2008 valuation
case_data = pd.DataFrame({
    "origin": exh3_years,
    "valuation": 2008,
    "case": exh3_case,
})
case_tri = cl.Triangle(
    case_data,
    origin="origin",
    valuation="valuation",
    columns="case",
    cumulative=True,
).val_to_dev()

# Benchmark reported and paid cumulative development patterns
# Development ages: 1998 at 132m down to 2003 at 72m
benchmark_reported = {
    132: 1.015,
    120: 1.020,
    108: 1.030,
    96: 1.051,
    84: 1.077,
    72: 1.131,
}
benchmark_paid = {
    132: 1.046,
    120: 1.067,
    108: 1.109,
    96: 1.187,
    84: 1.306,
    72: 1.489,
}

# Native ImpliedCaseDevelopment estimator (Friedland Approach #2)
implied_case = cl.ImpliedCaseDevelopment(
    reported_pattern=benchmark_reported,
    paid_pattern=benchmark_paid,
    style="cdf",
).fit(case_tri)

# Chainladder projection using implied case development factors
cl_case = cl.Chainladder().fit(implied_case.transform(case_tri))

# Extract implied case CDF factors and projected unpaid claims
ages = [132, 120, 108, 96, 84, 72]
cdf_frame = implied_case.case_cdf_.to_frame()
case_cdfs = [cdf_frame.loc["(All)", f"{age}-Ult"] for age in ages]
unpaid_estimates = (
    cl_case.ultimate_
    .to_frame(origin_as_datetime=False)
    .loc[[str(yr) for yr in exh3_years]]
    .iloc[:, 0]
    .values
)

exhibit_iii_df = pd.DataFrame(
    {
        "Case Outstanding (2)": exh3_case,
        "Reported CDF (3)": [benchmark_reported[a] for a in ages],
        "Paid CDF (4)": [benchmark_paid[a] for a in ages],
        "Case CDF (5)": case_cdfs,
        "Unpaid Estimate (6)": unpaid_estimates,
    },
    index=exh3_years,
)

# Summary row for display
exhibit_iii_display = exhibit_iii_df.copy()
exhibit_iii_display.index = exhibit_iii_display.index.astype(str)
exhibit_iii_display.loc["Total", "Case Outstanding (2)"] = exhibit_iii_df[
    "Case Outstanding (2)"
].sum()
exhibit_iii_display.loc["Total", "Unpaid Estimate (6)"] = exhibit_iii_df[
    "Unpaid Estimate (6)"
].sum()

print("Exhibit III: Self-Insurer Case Outstanding Only ($):")
nb_display(
    exhibit_iii_display.style.format(
        {
            "Case Outstanding (2)": "{:,.0f}",
            "Reported CDF (3)": "{:.3f}",
            "Paid CDF (4)": "{:.3f}",
            "Case CDF (5)": "{:.3f}",
            "Unpaid Estimate (6)": "{:,.0f}",
        },
        na_rep="",
    )
)

Exhibit III: Self-Insurer Case Outstanding Only ($):


,Case Outstanding (2),Reported CDF (3),Paid CDF (4),Case CDF (5),Unpaid Estimate (6)
1998,"500,000",1.015,1.046,1.506,"753,065"
1999,"650,000",1.020,1.067,1.454,"945,128"
2000,"800,000",1.030,1.109,1.421,"1,136,911"
2001,"850,000",1.051,1.187,1.445,"1,228,356"
2002,"975,000",1.077,1.306,1.439,"1,403,157"
2003,"1,000,000",1.131,1.489,1.545,"1,544,858"
Total,"4,775,000",,,,"7,011,474"


---

## Reconciliation & Validation Assertions

We validate our dynamic calculations against the printed textbook benchmarks using standard actuarial tolerances (`rtol=1e-3` / `atol=1e-3`) to verify zero hardcoding while confirming exact mathematical reconciliation.


In [ ]:
# Exhibit I: Selected Ratios & Ultimate
assert np.isclose(selected_paid_ratios[24], 0.833, atol=1e-3)
assert np.isclose(selected_case_ratios[24], 0.526, atol=1e-3)
assert np.isclose(sheet5_df["Projected Ultimate"].sum(), 567757738, rtol=1e-3)

# Exhibit II: Selected Ratios & Ultimate (XYZ Insurer)
assert np.isclose(xyz_selected_paid[24], 0.437, atol=1e-3)
assert np.isclose(xyz_selected_case[24], 1.367, atol=1e-3)
assert np.isclose(
    xyz_sheet5_df["Projected Ultimate"].loc[pd.Timestamp("2000-12-31")],
    37145,
    rtol=1e-3,
)
assert np.isclose(
    xyz_sheet5_df["Projected Ultimate"].loc[pd.Timestamp("2002-12-31")],
    48796,
    rtol=1e-3,
)
assert np.isclose(
    xyz_sheet5_df["Projected Ultimate"].loc[pd.Timestamp("2003-12-31")],
    45093,
    rtol=1e-3,
)
assert np.isclose(
    xyz_sheet5_df["Projected Ultimate"].loc[pd.Timestamp("2007-12-31")],
    46198,
    rtol=1e-3,
)
assert np.isclose(
    xyz_sheet5_df["Projected Ultimate"].loc[pd.Timestamp("2008-12-31")],
    46005,
    rtol=1e-3,
)
# Reconciles continuous calculation 514,181 and matches textbook 513,944 within 1e-3
assert np.isclose(xyz_sheet5_df["Projected Ultimate"].sum(), 514181, rtol=1e-3)
assert np.isclose(xyz_sheet5_df["Projected Ultimate"].sum(), 513944, rtol=1e-3)

# Exhibit III: Approach #2
assert np.isclose(exhibit_iii_df.loc[1998, "Case CDF (5)"], 1.506, atol=1e-3)
assert np.isclose(exhibit_iii_df.loc[2003, "Case CDF (5)"], 1.545, atol=1e-3)
assert np.isclose(exhibit_iii_df["Unpaid Estimate (6)"].sum(), 7011175, rtol=1e-3)
assert np.isclose(cl_case.ultimate_.sum().sum(), 7011175, rtol=1e-3)

print("All reconciliation assertions successfully passed!")